# MedicalNLP-RAG — Phase 4A
## LLM Answer-Generation Baselines

This notebook starts answer generation after freezing the retrieval pipeline:

`BM25 + MedCPT → RRF → MedCPT Cross-Encoder`

The **same LLM** is evaluated under three evidence conditions:

1. `closed_book` — question only.
2. `gold_evidence` — question + BioASQ expert gold snippets.
3. `retrieved_rag` — question + top documents from the frozen retrieval pipeline.

This isolates the effect of evidence access.

### Model
Default: `Qwen/Qwen3-4B-Instruct-2507`

The model is run locally in Colab with deterministic greedy decoding and 4-bit NF4 quantization.

### Output schema
Every answer must be JSON:

```json
{"exact_answer": "... or list ...", "ideal_answer": "..."}
```

Rules:
- yesno → exact answer `"yes"` or `"no"`
- factoid → list of up to 5 short answers
- list → list of short answer items
- summary → `[]`
- ideal answer → concise paragraph, maximum 150 words

### Safety
Only the **862 validation questions** are used.
The 860-question internal test remains locked.

**First run only the 12-question smoke test.**
Full 862×3 generation is disabled by default.

## 0. Configuration

In [ ]:
PROJECT_NAME = "MedicalNLP_RAG"
MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"
SEED = 42

USE_4BIT = True
BATCH_SIZE = 2

RAG_TOP_K = 10
MAX_EVIDENCE_TOKENS = 6000
MAX_INPUT_TOKENS = 8192
MAX_NEW_TOKENS = 320

RUN_SMOKE_TEST = True
RUN_FULL_GENERATION = False
SMOKE_PER_TYPE = 3

CONDITIONS = ["closed_book", "gold_evidence", "retrieved_rag"]

ALLOW_INTERNAL_TEST_EVALUATION = False

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2. Install/check dependencies

In [ ]:
import sys, subprocess, importlib, importlib.util
from packaging.version import Version

def get_version(name):
    try:
        return getattr(importlib.import_module(name), "__version__", "0")
    except Exception:
        return None

tv = get_version("transformers")

need_install = (
    tv is None
    or Version(tv) < Version("4.51.0")
    or importlib.util.find_spec("bitsandbytes") is None
    or importlib.util.find_spec("accelerate") is None
    or importlib.util.find_spec("rouge_score") is None
)

if need_install:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "transformers>=4.51.0",
        "accelerate>=1.0.0",
        "bitsandbytes>=0.45.0",
        "rouge-score>=0.1.2"
    ])

print("Dependency check complete.")

## 3. Imports and paths

In [ ]:
from pathlib import Path
import os, re, json, random, unicodedata, platform, warnings
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sklearn.metrics import f1_score
from rouge_score import rouge_scorer
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

ROOT = Path(f"/content/drive/MyDrive/{PROJECT_NAME}")
SPLIT_DIR = ROOT / "data" / "bioasq14_splits"
CACHE_DIR = ROOT / "data" / "pubmed_cache"

PHASE3C_DIR = ROOT / "results" / "phase3C_reranking"
PHASE3C_TABLE_DIR = PHASE3C_DIR / "tables"

PHASE4A_DIR = ROOT / "results" / "phase4A_answer_generation"
TABLE_DIR = PHASE4A_DIR / "tables"
FIG_DIR = PHASE4A_DIR / "figures"
GEN_DIR = PHASE4A_DIR / "generations"
LOG_DIR = PHASE4A_DIR / "logs"

for p in [PHASE4A_DIR, TABLE_DIR, FIG_DIR, GEN_DIR, LOG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

VALIDATION_JSON = SPLIT_DIR / "validation.json"
CORPUS_CSV = CACHE_DIR / "pubmed_closed_corpus.csv"
FROZEN_RUN = PHASE3C_TABLE_DIR / "RRF_MedCPT_CrossEncoder_top100.jsonl"
FROZEN_SELECTION = PHASE3C_DIR / "frozen_retrieval_pipeline.json"

assert VALIDATION_JSON.exists()
assert CORPUS_CSV.exists()
assert FROZEN_RUN.exists()
assert FROZEN_SELECTION.exists()

print("Phase 4A output:", PHASE4A_DIR)

## 4. Verify frozen retrieval

In [ ]:
with open(FROZEN_SELECTION, "r", encoding="utf-8") as f:
    frozen_selection = json.load(f)

print(json.dumps(frozen_selection, indent=2))

assert frozen_selection["frozen_pipeline"] == "RRF+MedCPT_CrossEncoder"
assert abs(
    frozen_selection["crossencoder_recall10"] - 0.6940663612120019
) < 1e-10

print("Frozen retrieval pipeline verified.")

## 5. GPU check

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("Enable a Colab GPU first.")

print("GPU:", torch.cuda.get_device_name(0))
props = torch.cuda.get_device_properties(0)
print("VRAM:", round(props.total_memory / (1024**3), 2), "GB")

## 6. Load validation data

In [ ]:
with open(VALIDATION_JSON, "r", encoding="utf-8") as f:
    validation_questions = json.load(f)["questions"]

assert len(validation_questions) == 862

rows = []
for q in validation_questions:
    rows.append({
        "id": str(q["id"]),
        "type": str(q["type"]).strip().lower(),
        "body": str(q["body"]).strip(),
        "exact_answer_gold": q.get("exact_answer"),
        "ideal_answer_gold": q.get("ideal_answer"),
        "snippets_gold": q.get("snippets", []),
        "documents_gold": q.get("documents", []),
    })

val_df = pd.DataFrame(rows)

assert val_df["id"].nunique() == 862
assert set(val_df["type"]) == {"yesno", "factoid", "list", "summary"}

display(val_df["type"].value_counts())

## 7. Load corpus and frozen rankings

In [ ]:
corpus_df = pd.read_csv(CORPUS_CSV, dtype={"pmid": str})
corpus_df["pmid"] = corpus_df["pmid"].astype(str).str.replace(r"\.0$", "", regex=True)
corpus_df["title"] = corpus_df["title"].fillna("").astype(str).str.strip()
corpus_df["abstract"] = corpus_df["abstract"].fillna("").astype(str).str.strip()

corpus_lookup = (
    corpus_df.drop_duplicates("pmid")
    .set_index("pmid")[["title", "abstract"]]
    .to_dict(orient="index")
)

frozen_rankings = {}
with open(FROZEN_RUN, "r", encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line)
        frozen_rankings[str(rec["id"])] = [str(x) for x in rec["ranked_pmids"]]

assert len(frozen_rankings) == 862
assert all(len(v) >= RAG_TOP_K for v in frozen_rankings.values())

question_lookup = {str(q["id"]): q for q in validation_questions}

print("Corpus records:", len(corpus_lookup))
print("Frozen rankings:", len(frozen_rankings))

## 8. Experiment environment

In [ ]:
import transformers

environment = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "model": MODEL_NAME,
    "seed": SEED,
    "quantization": "4bit_NF4" if USE_4BIT else "none",
    "batch_size": BATCH_SIZE,
    "rag_top_k": RAG_TOP_K,
    "max_evidence_tokens": MAX_EVIDENCE_TOKENS,
    "max_input_tokens": MAX_INPUT_TOKENS,
    "max_new_tokens": MAX_NEW_TOKENS,
    "decoding": "greedy_do_sample_false",
    "validation_questions": 862,
    "internal_test_evaluated": False,
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "gpu": torch.cuda.get_device_name(0),
}

with open(LOG_DIR / "environment.json", "w", encoding="utf-8") as f:
    json.dump(environment, f, indent=2, ensure_ascii=False)

print(json.dumps(environment, indent=2, ensure_ascii=False))

## 9. Load Qwen3-4B-Instruct-2507

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "left"

if USE_4BIT:
    qconfig = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=qconfig,
        device_map="auto",
        torch_dtype=torch.float16,
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        device_map="auto",
        torch_dtype="auto",
    )

model.eval()
model.generation_config.pad_token_id = tokenizer.pad_token_id
model.generation_config.eos_token_id = tokenizer.eos_token_id

print("Loaded:", MODEL_NAME)

## 10. Evidence packing

In [ ]:
def token_count(text):
    return len(tokenizer(text, add_special_tokens=False)["input_ids"])

def truncate_to_tokens(text, max_tokens):
    ids = tokenizer(text, add_special_tokens=False)["input_ids"]
    if len(ids) <= max_tokens:
        return text
    return tokenizer.decode(ids[:max_tokens], skip_special_tokens=True)

def pack_blocks(blocks, max_tokens=MAX_EVIDENCE_TOKENS):
    packed = []
    used = 0

    for block in blocks:
        block = str(block).strip()
        if not block:
            continue

        n = token_count(block)

        if used + n <= max_tokens:
            packed.append(block)
            used += n
        else:
            remaining = max_tokens - used
            if remaining > 32:
                packed.append(truncate_to_tokens(block, remaining))
                used = max_tokens
            break

    return "\n\n".join(packed), used

## 11. Gold evidence builder

In [ ]:
def build_gold_evidence(question_record):
    blocks = []

    for i, snippet in enumerate(question_record.get("snippets", []), start=1):
        if isinstance(snippet, dict):
            text = str(snippet.get("text", "")).strip()
            document = str(snippet.get("document", "")).strip()
        else:
            text = str(snippet).strip()
            document = ""

        if not text:
            continue

        block = f"[Gold snippet {i}]"
        if document:
            block += f" Source: {document}"
        block += f"\n{text}"
        blocks.append(block)

    return pack_blocks(blocks)

## 12. Retrieved RAG evidence builder

In [ ]:
def build_retrieved_evidence(qid):
    blocks = []

    for rank, pmid in enumerate(frozen_rankings[qid][:RAG_TOP_K], start=1):
        rec = corpus_lookup.get(pmid)
        if rec is None:
            continue

        blocks.append(
            f"[Retrieved document {rank}] PMID: {pmid}\n"
            f"Title: {rec['title']}\n"
            f"Abstract: {rec['abstract']}"
        )

    return pack_blocks(blocks)

## 13. Evidence token audit

In [ ]:
audit_rows = []

for row in tqdm(val_df.itertuples(index=False), total=len(val_df)):
    _, gold_tokens = build_gold_evidence(question_lookup[row.id])
    _, rag_tokens = build_retrieved_evidence(row.id)

    audit_rows.append({
        "id": row.id,
        "type": row.type,
        "gold_evidence_tokens": gold_tokens,
        "retrieved_evidence_tokens": rag_tokens,
    })

evidence_audit = pd.DataFrame(audit_rows)
evidence_audit.to_csv(TABLE_DIR / "evidence_token_audit.csv", index=False)

display(
    evidence_audit.groupby("type")[
        ["gold_evidence_tokens", "retrieved_evidence_tokens"]
    ].describe().round(1)
)

## 14. Fixed benchmark prompt

In [ ]:
SYSTEM_PROMPT = """You are a biomedical question-answering model used in a research benchmark.

Rules:
1. Return JSON only. Do not use Markdown or code fences.
2. Do not provide hidden reasoning or chain-of-thought.
3. Follow the required output shape exactly.
4. The ideal_answer must be concise, factual, and at most 150 words.
5. Do not invent citations or PMID identifiers.
"""

FORMAT_RULES = {
    "yesno": """Question type: yesno.
Return exactly this JSON structure:
{"exact_answer":"yes","ideal_answer":"concise biomedical explanation"}
or use "no" as exact_answer.
The exact_answer must be exactly yes or no.""",

    "factoid": """Question type: factoid.
Return exactly this JSON structure:
{"exact_answer":["answer 1","answer 2"],"ideal_answer":"concise biomedical explanation"}
The exact_answer must contain at most 5 short candidate answers, ordered from most to least likely.""",

    "list": """Question type: list.
Return exactly this JSON structure:
{"exact_answer":["item 1","item 2"],"ideal_answer":"concise biomedical explanation"}
The exact_answer must be a JSON list of distinct short items.""",

    "summary": """Question type: summary.
Return exactly this JSON structure:
{"exact_answer":[],"ideal_answer":"concise biomedical summary"}
The exact_answer must be an empty JSON list."""
}

def build_messages(question, qtype, condition, evidence=""):
    if condition == "closed_book":
        evidence_rule = (
            "No external evidence is provided. "
            "Answer using the model's internal biomedical knowledge."
        )
    elif condition == "gold_evidence":
        evidence_rule = (
            "Expert-selected BioASQ evidence is provided. "
            "Base the answer on this evidence and do not add unsupported claims."
        )
    elif condition == "retrieved_rag":
        evidence_rule = (
            "Automatically retrieved PubMed evidence is provided. "
            "Base the answer on this evidence and do not add unsupported claims."
        )
    else:
        raise ValueError(condition)

    parts = [
        FORMAT_RULES[qtype],
        evidence_rule,
        f"Question: {question}",
    ]

    if evidence:
        parts.append("Evidence:\\n" + evidence)

    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": "\\n\\n".join(parts)},
    ]

## 15. JSON parsing and validation

In [ ]:
def clean_model_text(text):
    text = str(text).strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\\s*", "", text, flags=re.IGNORECASE)
        text = re.sub(r"\\s*```$", "", text)
    return text.strip()

def extract_json_object(text):
    text = clean_model_text(text)

    try:
        return json.loads(text), "direct"
    except Exception:
        pass

    start = text.find("{")
    end = text.rfind("}")

    if start >= 0 and end > start:
        try:
            return json.loads(text[start:end+1]), "extracted"
        except Exception:
            pass

    return None, "invalid"

def validate_prediction_object(obj, qtype):
    if not isinstance(obj, dict):
        return False, "not_dict"

    if "exact_answer" not in obj or "ideal_answer" not in obj:
        return False, "missing_required_field"

    if not isinstance(obj["ideal_answer"], str):
        return False, "ideal_not_string"

    exact = obj["exact_answer"]

    if qtype == "yesno":
        if not isinstance(exact, str):
            return False, "yesno_exact_not_string"
        if exact.strip().lower() not in {"yes", "no"}:
            return False, "yesno_invalid_value"

    elif qtype in {"factoid", "list"}:
        if not isinstance(exact, list):
            return False, "exact_not_list"
        if not all(isinstance(x, str) for x in exact):
            return False, "exact_list_nonstring"
        if qtype == "factoid" and len(exact) > 5:
            return False, "factoid_more_than_5"

    elif qtype == "summary":
        if not isinstance(exact, list) or len(exact) != 0:
            return False, "summary_exact_not_empty"

    return True, "valid"

def parse_prediction(raw_text, qtype):
    obj, parse_mode = extract_json_object(raw_text)

    if obj is None:
        return {
            "parse_valid": False,
            "parse_mode": parse_mode,
            "validation_status": "invalid_json",
            "exact_answer_pred": None,
            "ideal_answer_pred": "",
        }

    valid, status = validate_prediction_object(obj, qtype)

    return {
        "parse_valid": bool(valid),
        "parse_mode": parse_mode,
        "validation_status": status,
        "exact_answer_pred": obj.get("exact_answer"),
        "ideal_answer_pred": (
            obj.get("ideal_answer", "")
            if isinstance(obj.get("ideal_answer", ""), str)
            else ""
        ),
    }

## 16. Prepare generation jobs

In [ ]:
def prepare_generation_row(row, condition):
    qid = str(row["id"])

    if condition == "closed_book":
        evidence = ""
        evidence_tokens = 0
    elif condition == "gold_evidence":
        evidence, evidence_tokens = build_gold_evidence(question_lookup[qid])
    elif condition == "retrieved_rag":
        evidence, evidence_tokens = build_retrieved_evidence(qid)
    else:
        raise ValueError(condition)

    messages = build_messages(
        question=row["body"],
        qtype=row["type"],
        condition=condition,
        evidence=evidence,
    )

    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    return {
        "id": qid,
        "type": row["type"],
        "question": row["body"],
        "condition": condition,
        "evidence_tokens": int(evidence_tokens),
        "prompt_text": prompt_text,
    }

## 17. Resumable deterministic generation

In [ ]:
def cache_path(run_name):
    return GEN_DIR / f"{run_name}.jsonl"

def load_existing(path):
    out = {}
    if not path.exists():
        return out

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            rec = json.loads(line)
            out[(str(rec["id"]), str(rec["condition"]))] = rec

    return out

def append_jsonl(path, records):
    with open(path, "a", encoding="utf-8") as f:
        for rec in records:
            f.write(json.dumps(rec, ensure_ascii=False, default=str) + "\n")
        f.flush()

def generate_dataframe(input_df, run_name):
    path = cache_path(run_name)
    existing = load_existing(path)

    jobs = []

    for _, row in input_df.iterrows():
        for condition in CONDITIONS:
            key = (str(row["id"]), condition)
            if key not in existing:
                jobs.append(prepare_generation_row(row, condition))

    print("Run:", run_name)
    print("Existing:", len(existing))
    print("Remaining:", len(jobs))

    for start in tqdm(range(0, len(jobs), BATCH_SIZE), desc=run_name):
        batch_jobs = jobs[start:start+BATCH_SIZE]
        prompts = [x["prompt_text"] for x in batch_jobs]

        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_INPUT_TOKENS,
        )

        inputs = {k: v.to(model.device) for k, v in inputs.items()}
        input_width = inputs["input_ids"].shape[1]

        with torch.inference_mode():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                use_cache=True,
            )

        generated_only = output_ids[:, input_width:]
        decoded = tokenizer.batch_decode(generated_only, skip_special_tokens=True)

        records = []

        for job, raw in zip(batch_jobs, decoded):
            parsed = parse_prediction(raw, job["type"])

            records.append({
                "id": job["id"],
                "type": job["type"],
                "condition": job["condition"],
                "question": job["question"],
                "evidence_tokens": job["evidence_tokens"],
                "raw_output": raw,
                **parsed,
            })

        append_jsonl(path, records)

    complete = load_existing(path)
    requested = set(input_df["id"].astype(str))

    rows = [
        rec for rec in complete.values()
        if str(rec["id"]) in requested
    ]

    return pd.DataFrame(rows)

## 18. Build 12-question stratified smoke sample

In [ ]:
smoke_parts = []

for qtype in sorted(val_df["type"].unique()):
    smoke_parts.append(
        val_df[val_df["type"] == qtype].sample(
            n=SMOKE_PER_TYPE,
            random_state=SEED
        )
    )

smoke_df = (
    pd.concat(smoke_parts, ignore_index=True)
    .sort_values(["type", "id"])
    .reset_index(drop=True)
)

print("Smoke questions:", len(smoke_df))
display(smoke_df[["id", "type", "body"]])

## 19. Run smoke test

In [ ]:
smoke_results = None

if RUN_SMOKE_TEST:
    smoke_results = generate_dataframe(
        smoke_df,
        "smoke_12q_3conditions"
    )

    smoke_results.to_csv(
        TABLE_DIR / "smoke_12q_3conditions.csv",
        index=False
    )

    display(
        smoke_results.groupby(["condition", "type"])["parse_valid"]
        .agg(["count", "mean"])
    )
else:
    print("Smoke test skipped.")

## 20. Inspect smoke outputs

In [ ]:
if smoke_results is not None:
    display(
        smoke_results[
            [
                "id",
                "type",
                "condition",
                "parse_valid",
                "validation_status",
                "exact_answer_pred",
                "ideal_answer_pred",
            ]
        ].sort_values(["type", "id", "condition"])
    )

# STOP POINT 4A-SMOKE

At this point **stop and inspect the outputs**.

We want:
- valid JSON in almost all 36 generations;
- correct exact-answer shape for all four question types;
- concise ideal answers;
- no obvious prompt failure.

Leave:

```python
RUN_FULL_GENERATION = False
```

until the smoke output is reviewed.

The rest of the notebook is already prepared for the full run.

In [ ]:
RUN_FULL_GENERATION = True
print("Full generation enabled:", RUN_FULL_GENERATION)

## 21. Answer normalization helpers

In [ ]:
def normalize_answer_text(text):
    text = unicodedata.normalize("NFKC", str(text))
    text = text.lower().strip()
    text = re.sub(r"[_]+", " ", text)
    text = re.sub(r"[^\w\s\.\-\+/]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def as_gold_groups(exact_answer, qtype):
    if qtype == "yesno":
        return [[normalize_answer_text(exact_answer)]] if exact_answer is not None else []

    if qtype not in {"factoid", "list"}:
        return []

    if exact_answer is None:
        return []

    if not isinstance(exact_answer, list):
        exact_answer = [exact_answer]

    groups = []

    for item in exact_answer:
        if isinstance(item, list):
            syns = [normalize_answer_text(x) for x in item if str(x).strip()]
        else:
            syns = [normalize_answer_text(item)]

        syns = [x for x in syns if x]
        if syns:
            groups.append(syns)

    return groups

def prediction_list(pred):
    if isinstance(pred, list):
        return [str(x).strip() for x in pred if str(x).strip()]
    if isinstance(pred, str) and pred.strip():
        return [pred.strip()]
    return []

## 22. Exact-answer scoring

In [ ]:
def score_factoid(pred, gold):
    preds = prediction_list(pred)[:5]
    groups = as_gold_groups(gold, "factoid")

    all_gold = {syn for group in groups for syn in group}

    norm_preds = [normalize_answer_text(x) for x in preds]

    strict = float(bool(norm_preds) and norm_preds[0] in all_gold)

    first_rank = None
    for rank, p in enumerate(norm_preds, start=1):
        if p in all_gold:
            first_rank = rank
            break

    return {
        "factoid_strict": strict,
        "factoid_lenient": float(first_rank is not None),
        "factoid_mrr": 1.0 / first_rank if first_rank else 0.0,
    }

def score_list(pred, gold):
    preds = prediction_list(pred)
    groups = as_gold_groups(gold, "list")

    unmatched = set(range(len(groups)))
    matched = 0

    for pred_item in preds:
        p = normalize_answer_text(pred_item)

        for gi in list(unmatched):
            if p in set(groups[gi]):
                matched += 1
                unmatched.remove(gi)
                break

    precision = matched / len(preds) if preds else 0.0
    recall = matched / len(groups) if groups else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall > 0
        else 0.0
    )

    return {
        "list_precision": precision,
        "list_recall": recall,
        "list_f1": f1,
    }

## 23. Ideal-answer ROUGE

In [ ]:
rouge = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True
)

def get_gold_ideal(value):
    if value is None:
        return ""
    if isinstance(value, list):
        return str(value[0]).strip() if value else ""
    return str(value).strip()

def score_ideal(pred, gold):
    pred = str(pred or "").strip()
    gold = get_gold_ideal(gold)

    if not pred or not gold:
        return {
            "rouge1_f": np.nan,
            "rouge2_f": np.nan,
            "rougeL_f": np.nan,
        }

    s = rouge.score(gold, pred)

    return {
        "rouge1_f": s["rouge1"].fmeasure,
        "rouge2_f": s["rouge2"].fmeasure,
        "rougeL_f": s["rougeL"].fmeasure,
    }

## 24. Evaluate generation results

In [ ]:
gold_lookup = (
    val_df.set_index("id")[
        ["type", "exact_answer_gold", "ideal_answer_gold"]
    ].to_dict(orient="index")
)

def evaluate_generation_results(results_df):
    scored_rows = []

    for row in results_df.itertuples(index=False):
        gold = gold_lookup[str(row.id)]
        qtype = row.type

        rec = {
            "id": str(row.id),
            "type": qtype,
            "condition": row.condition,
            "parse_valid": bool(row.parse_valid),
            "yesno_correct": np.nan,
            "factoid_strict": np.nan,
            "factoid_lenient": np.nan,
            "factoid_mrr": np.nan,
            "list_precision": np.nan,
            "list_recall": np.nan,
            "list_f1": np.nan,
        }

        if row.parse_valid:
            if qtype == "yesno":
                rec["yesno_correct"] = float(
                    normalize_answer_text(row.exact_answer_pred)
                    == normalize_answer_text(gold["exact_answer_gold"])
                )
            elif qtype == "factoid":
                rec.update(score_factoid(
                    row.exact_answer_pred,
                    gold["exact_answer_gold"]
                ))
            elif qtype == "list":
                rec.update(score_list(
                    row.exact_answer_pred,
                    gold["exact_answer_gold"]
                ))

        rec.update(score_ideal(
            row.ideal_answer_pred,
            gold["ideal_answer_gold"]
        ))

        scored_rows.append(rec)

    scored = pd.DataFrame(scored_rows)
    summaries = []

    for condition, group in scored.groupby("condition"):
        summary = {
            "condition": condition,
            "n": len(group),
            "json_valid_rate": float(group["parse_valid"].mean()),
            "rouge1_f": float(group["rouge1_f"].mean()),
            "rouge2_f": float(group["rouge2_f"].mean()),
            "rougeL_f": float(group["rougeL_f"].mean()),
        }

        yes = group[group["type"] == "yesno"]
        if len(yes):
            summary["yesno_accuracy"] = float(yes["yesno_correct"].mean())

            y_true, y_pred = [], []
            for qid in yes["id"]:
                g = normalize_answer_text(gold_lookup[str(qid)]["exact_answer_gold"])

                row_match = results_df[
                    (results_df["condition"] == condition)
                    & (results_df["id"].astype(str) == str(qid))
                ].iloc[0]

                p = (
                    normalize_answer_text(row_match["exact_answer_pred"])
                    if bool(row_match["parse_valid"])
                    else "__invalid__"
                )

                y_true.append(g)
                y_pred.append(p)

            summary["yesno_macro_f1"] = float(
                f1_score(
                    y_true,
                    y_pred,
                    labels=["yes", "no"],
                    average="macro",
                    zero_division=0
                )
            )

        fact = group[group["type"] == "factoid"]
        if len(fact):
            summary["factoid_strict"] = float(fact["factoid_strict"].mean())
            summary["factoid_lenient"] = float(fact["factoid_lenient"].mean())
            summary["factoid_mrr"] = float(fact["factoid_mrr"].mean())

        lst = group[group["type"] == "list"]
        if len(lst):
            summary["list_precision"] = float(lst["list_precision"].mean())
            summary["list_recall"] = float(lst["list_recall"].mean())
            summary["list_f1"] = float(lst["list_f1"].mean())

        summaries.append(summary)

    return scored, pd.DataFrame(summaries)

# Part G — Full validation run

After the smoke test is accepted, change:

```python
RUN_FULL_GENERATION = True
```

Then run the cells below.

Total generations: **862 × 3 = 2,586**.

The run is checkpointed and resumable.

## 25. Run full validation generation

In [ ]:
full_results = None

if RUN_FULL_GENERATION:
    full_results = generate_dataframe(
        val_df,
        "validation_862q_3conditions"
    )

    assert len(full_results) == 862 * 3

    full_results.to_csv(
        TABLE_DIR / "validation_862q_3conditions.csv",
        index=False
    )

    print("Full generation complete:", len(full_results))
else:
    print("RUN_FULL_GENERATION=False — full run not started.")

## 26. JSON compliance

In [ ]:
if full_results is not None:
    compliance = (
        full_results.groupby(["condition", "type"])["parse_valid"]
        .agg(["count", "mean"])
        .reset_index()
    )

    compliance.to_csv(
        TABLE_DIR / "json_compliance_by_condition_type.csv",
        index=False
    )

    display(compliance)

## 27. Automatic validation scores

In [ ]:
full_scored = None
full_summary = None

if full_results is not None:
    full_scored, full_summary = evaluate_generation_results(full_results)

    full_scored.to_csv(
        TABLE_DIR / "validation_answer_scores_per_question.csv",
        index=False
    )

    full_summary.to_csv(
        TABLE_DIR / "validation_answer_generation_summary.csv",
        index=False
    )

    display(full_summary)

## 28. Ideal-answer ROUGE figure

In [ ]:
if full_summary is not None:
    plot_df = full_summary[
        ["condition", "rouge1_f", "rouge2_f", "rougeL_f"]
    ].set_index("condition")

    fig, ax = plt.subplots(figsize=(8, 5))
    plot_df.plot(kind="bar", ax=ax)

    ax.set_xlabel("Evidence condition")
    ax.set_ylabel("ROUGE F1")
    ax.set_title("Ideal-answer automatic evaluation")
    ax.set_ylim(0, 1)
    ax.tick_params(axis="x", rotation=0)

    fig.tight_layout()
    fig.savefig(
        FIG_DIR / "fig01_ideal_answer_rouge.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.show()

## 29. Exact-answer metric figure

In [ ]:
if full_summary is not None:
    metric_rows = []

    for row in full_summary.to_dict(orient="records"):
        for metric in [
            "yesno_accuracy",
            "yesno_macro_f1",
            "factoid_strict",
            "factoid_lenient",
            "factoid_mrr",
            "list_f1",
        ]:
            if metric in row and pd.notna(row[metric]):
                metric_rows.append({
                    "condition": row["condition"],
                    "metric": metric,
                    "score": row[metric],
                })

    metric_df = pd.DataFrame(metric_rows)
    pivot = metric_df.pivot(
        index="metric",
        columns="condition",
        values="score"
    )

    fig, ax = plt.subplots(figsize=(9, 5.5))
    pivot.plot(kind="bar", ax=ax)

    ax.set_xlabel("Metric")
    ax.set_ylabel("Score")
    ax.set_title("BioASQ exact-answer development metrics")
    ax.set_ylim(0, 1)
    ax.tick_params(axis="x", rotation=35)

    fig.tight_layout()
    fig.savefig(
        FIG_DIR / "fig02_exact_answer_metrics.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.show()

## 30. Paired bootstrap helper

In [ ]:
def paired_bootstrap_from_scored(
    scored_df,
    condition_a,
    condition_b,
    metric,
    qtype=None,
    n_boot=5000,
    seed=SEED,
):
    df = scored_df.copy()

    if qtype is not None:
        df = df[df["type"] == qtype]

    a = df[df["condition"] == condition_a][["id", metric]].rename(columns={metric: "a"})
    b = df[df["condition"] == condition_b][["id", metric]].rename(columns={metric: "b"})

    merged = a.merge(b, on="id", validate="one_to_one").dropna()

    if len(merged) == 0:
        return None

    va = merged["a"].to_numpy(float)
    vb = merged["b"].to_numpy(float)

    rng = np.random.default_rng(seed)
    n = len(merged)
    deltas = []

    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        deltas.append(np.mean(va[idx]) - np.mean(vb[idx]))

    deltas = np.asarray(deltas)

    return {
        "condition_a": condition_a,
        "condition_b": condition_b,
        "metric": metric,
        "qtype": qtype or "all",
        "n": int(n),
        "mean_delta": float(deltas.mean()),
        "ci95_low": float(np.quantile(deltas, 0.025)),
        "ci95_high": float(np.quantile(deltas, 0.975)),
        "bootstrap_probability_delta_gt_0": float(np.mean(deltas > 0)),
    }

## 31. Paired bootstrap comparisons

In [ ]:
if full_scored is not None:
    comparisons = [
        ("gold_evidence", "closed_book"),
        ("retrieved_rag", "closed_book"),
        ("gold_evidence", "retrieved_rag"),
    ]

    specs = [
        ("rougeL_f", None),
        ("yesno_correct", "yesno"),
        ("factoid_strict", "factoid"),
        ("factoid_lenient", "factoid"),
        ("factoid_mrr", "factoid"),
        ("list_f1", "list"),
    ]

    boot_rows = []

    for a, b in comparisons:
        for metric, qtype in specs:
            r = paired_bootstrap_from_scored(
                full_scored,
                a,
                b,
                metric,
                qtype=qtype
            )
            if r is not None:
                boot_rows.append(r)

    bootstrap_df = pd.DataFrame(boot_rows)

    bootstrap_df.to_csv(
        TABLE_DIR / "paired_bootstrap_answer_generation.csv",
        index=False
    )

    display(bootstrap_df)

## 32. Closed-book vs RAG disagreement analysis

In [ ]:
if full_scored is not None:
    metric_for_type = {
        "yesno": "yesno_correct",
        "factoid": "factoid_lenient",
        "list": "list_f1",
    }

    disagreement_rows = []

    for qtype, metric in metric_for_type.items():
        subset = full_scored[full_scored["type"] == qtype]

        pivot = subset.pivot(
            index="id",
            columns="condition",
            values=metric
        )

        for qid, r in pivot.iterrows():
            disagreement_rows.append({
                "id": qid,
                "type": qtype,
                "metric": metric,
                "closed_book": r.get("closed_book"),
                "gold_evidence": r.get("gold_evidence"),
                "retrieved_rag": r.get("retrieved_rag"),
            })

    disagreement_df = pd.DataFrame(disagreement_rows)

    disagreement_df["rag_minus_closed"] = (
        disagreement_df["retrieved_rag"]
        - disagreement_df["closed_book"]
    )
    disagreement_df["gold_minus_rag"] = (
        disagreement_df["gold_evidence"]
        - disagreement_df["retrieved_rag"]
    )

    body_lookup = val_df.set_index("id")["body"].to_dict()
    disagreement_df["question"] = disagreement_df["id"].map(body_lookup)

    disagreement_df.to_csv(
        TABLE_DIR / "answer_condition_disagreements.csv",
        index=False
    )

    print("Largest RAG improvements:")
    display(
        disagreement_df.sort_values(
            "rag_minus_closed",
            ascending=False
        ).head(25)
    )

    print("\nLargest RAG degradations:")
    display(
        disagreement_df.sort_values(
            "rag_minus_closed",
            ascending=True
        ).head(25)
    )

## 33. Phase 4A report

In [ ]:
if full_summary is not None:
    phase4a_report = {
        "dataset": "BioASQ14b",
        "validation_questions": 862,
        "internal_test_evaluated": False,
        "frozen_retrieval_pipeline": frozen_selection,
        "llm": {
            "model": MODEL_NAME,
            "quantization": "4bit_NF4" if USE_4BIT else "none",
            "decoding": "greedy",
            "do_sample": False,
            "max_input_tokens": MAX_INPUT_TOKENS,
            "max_new_tokens": MAX_NEW_TOKENS,
        },
        "rag": {
            "top_k": RAG_TOP_K,
            "max_evidence_tokens": MAX_EVIDENCE_TOKENS,
        },
        "conditions": CONDITIONS,
        "automatic_validation_results": full_summary.to_dict(orient="records"),
        "note": (
            "ROUGE is used only as an automatic development metric for ideal answers "
            "and does not replace BioASQ expert/manual assessment."
        ),
    }

    with open(
        PHASE4A_DIR / "phase4A_report.json",
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(phase4a_report, f, indent=2, ensure_ascii=False)

    print(json.dumps(phase4a_report, indent=2, ensure_ascii=False))

# STOP POINT 4A-FULL

After the full run, send:

1. JSON compliance table
2. answer-generation summary
3. ROUGE figure
4. exact-answer figure
5. paired-bootstrap table
6. examples where RAG improves closed-book
7. examples where RAG degrades closed-book
8. `phase4A_report.json`

Then Phase 4B will add **evidence filtering**.

Do not evaluate the locked internal test.

## 34. Locked-test safety

In [ ]:
if not ALLOW_INTERNAL_TEST_EVALUATION:
    print("INTERNAL TEST REMAINS LOCKED.")
else:
    raise RuntimeError("Do not unlock the internal test during Phase 4A.")

print("Phase 4A output:", PHASE4A_DIR)